Enrollment No.: 240840116035

Practical 10 :Mini Project: Analyze a real-world dataset, build an interactive dashboard, and present actionable insights with ethical considerations

### Cell 1 — Install required libraries

In [ ]:
!pip -q install pandas numpy plotly dash

### Cell 2 — Import libraries

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display
from google.colab import files
from dash import Dash, dcc, html, Input, Output
import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
print("Libraries imported successfully.")

Libraries imported successfully.


### Cell 3 — Upload a traffic accident dataset

In [ ]:
UPLOAD_REAL_CSV = False  # Change to True to upload your own CSV

uploaded_df = None
if UPLOAD_REAL_CSV:
    uploaded = files.upload()
    if uploaded:
        file_name = next(iter(uploaded))
        uploaded_df = pd.read_csv(file_name)
        print("Uploaded:", file_name)
        display(uploaded_df.head())
else:
    print("No CSV uploaded. Synthetic practice data will be created in Cell 4.")

No CSV uploaded. Synthetic practice data will be created in Cell 4.


### Cell 4 — Create a synthetic practice dataset if no CSV was uploaded

In [ ]:
if uploaded_df is None:
    np.random.seed(42)
    n = 1200
    dates = pd.to_datetime(np.random.choice(
        pd.date_range("2023-01-01", "2025-12-31", freq="D"), n
    ))
    cities = ["Ahmedabad", "Surat", "Vadodara", "Rajkot", "Gandhinagar"]
    road_types = ["Highway", "Main Road", "Residential", "Intersection"]
    weather_types = ["Clear", "Rainy", "Foggy", "Cloudy"]
    severity_types = ["Minor", "Moderate", "Severe", "Fatal"]
    city = np.random.choice(cities, n, p=[0.36, 0.22, 0.17, 0.15, 0.10])
    hour = np.random.choice(range(24), n, p=np.array(
        [2,1,1,1,1,2,4,7,8,5,4,4,4,4,4,5,7,9,9,7,5,4,3,2]
    ) / sum([2,1,1,1,1,2,4,7,8,5,4,4,4,4,4,5,7,9,9,7,5,4,3,2]))
    severity = np.random.choice(severity_types, n, p=[0.48, 0.32, 0.16, 0.04])
    df = pd.DataFrame({
        "Accident_ID": [f"ACC{i:05d}" for i in range(1, n + 1)],
        "Accident_Date": dates,
        "City": city,
        "Area": [f"Zone-{x}" for x in np.random.randint(1, 13, n)],
        "Road_Type": np.random.choice(road_types, n, p=[0.18, 0.42, 0.25, 0.15]),
        "Accident_Hour": hour,
        "Weather": np.random.choice(weather_types, n, p=[0.62, 0.16, 0.06, 0.16]),
        "Vehicle_Type": np.random.choice(["Two-wheeler", "Car", "Bus", "Truck", "Auto"], n,
                                         p=[0.40, 0.32, 0.08, 0.08, 0.12]),
        "Accident_Severity": severity,
        "Vehicles_Involved": np.random.choice([1, 2, 3, 4], n, p=[0.46, 0.36, 0.13, 0.05]),
        "Injuries": np.random.poisson(1.1, n),
        "Fatalities": np.random.choice([0, 1, 2], n, p=[0.96, 0.035, 0.005]),
        "Cause": np.random.choice(["Over-speeding", "Signal violation", "Poor road condition",
                                   "Distracted driving", "Weather", "Other"], n,
                                  p=[0.28, 0.20, 0.14, 0.18, 0.08, 0.12])
    })
    DATA_SOURCE = "Synthetic practice data — not official traffic or police records"
else:
    df = uploaded_df.copy()
    DATA_SOURCE = "User-uploaded CSV — verify and document its original source"

if uploaded_df is None:
    print("Data source:", DATA_SOURCE)
    print("Dataset shape:", df.shape)
    display(df.head())

Data source: Synthetic practice data — not official traffic or police records
Dataset shape: (1200, 13)


,Accident_ID,Accident_Date,City,Area,Road_Type,Accident_Hour,Weather,Vehicle_Type,Accident_Severity,Vehicles_Involved,Injuries,Fatalities,Cause
0,ACC00001,2025-05-10,Rajkot,Zone-1,Residential,17,Clear,Car,Minor,2,1,0,Over-speeding
1,ACC00002,2025-12-31,Gandhinagar,Zone-8,Residential,19,Clear,Two-wheeler,Severe,4,1,0,Over-speeding
2,ACC00003,2025-11-10,Ahmedabad,Zone-4,Highway,17,Clear,Two-wheeler,Minor,3,1,0,Signal violation
3,ACC00004,2023-05-02,Rajkot,Zone-4,Residential,6,Clear,Car,Minor,3,1,0,Over-speeding
4,ACC00005,2024-04-11,Surat,Zone-2,Residential,16,Rainy,Car,Minor,3,0,0,Over-speeding


### Cell 5 — Standardize column names and check required fields

In [ ]:
rename_map = {
    # Example mappings:
    # "date": "Accident_Date",
    # "time_hour": "Accident_Hour",
    # "severity": "Accident_Severity",
    # "district": "City"
}
df = df.rename(columns=rename_map)

required = ["Accident_Date", "City", "Accident_Hour", "Accident_Severity",
            "Road_Type", "Weather", "Vehicle_Type", "Cause"]
missing = [col for col in required if col not in df.columns]
if missing:
    raise ValueError(
        f"Missing required columns: {missing}. Edit rename_map or prepare your CSV."
    )

if "Accident_ID" not in df.columns:
    df["Accident_ID"] = [f"ROW{i:05d}" for i in range(1, len(df) + 1)]
for col in ["Area", "Injuries", "Fatalities", "Vehicles_Involved"]:
    if col not in df.columns:
        df[col] = np.nan

print("Column validation completed.")
print(df.columns.tolist())

Column validation completed.
['Accident_ID', 'Accident_Date', 'City', 'Area', 'Road_Type', 'Accident_Hour', 'Weather', 'Vehicle_Type', 'Accident_Severity', 'Vehicles_Involved', 'Injuries', 'Fatalities', 'Cause']


### Cell 6 — Clean the dataset

In [ ]:
df["Accident_Date"] = pd.to_datetime(df["Accident_Date"], errors="coerce")
for col in ["Accident_Hour", "Injuries", "Fatalities", "Vehicles_Involved"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

before = len(df)
df = df.drop_duplicates()
df = df.dropna(subset=["Accident_Date", "City", "Accident_Hour", "Accident_Severity"])
df["Accident_Hour"] = df["Accident_Hour"].clip(0, 23).astype(int)

text_cols = ["City", "Area", "Road_Type", "Weather", "Vehicle_Type",
             "Accident_Severity", "Cause"]
for col in text_cols:
    df[col] = df[col].astype("string").str.strip().str.title()

df["Month"] = df["Accident_Date"].dt.to_period("M").astype(str)
df["Year"] = df["Accident_Date"].dt.year
df["Weekday"] = df["Accident_Date"].dt.day_name()
df["Time_Period"] = pd.cut(
    df["Accident_Hour"], bins=[-1, 5, 11, 17, 23],
    labels=["Night (00-05)", "Morning (06-11)", "Afternoon (12-17)", "Evening (18-23)"]
)

print("Duplicate rows removed:", before - len(df))
print("Rows after cleaning:", len(df))
print("Missing values:")
display(df.isna().sum().to_frame("Missing values"))

Duplicate rows removed: 0
Rows after cleaning: 1200
Missing values:


,Missing values
Accident_ID,0
Accident_Date,0
City,0
Area,0
Road_Type,0
Accident_Hour,0
Weather,0
Vehicle_Type,0
Accident_Severity,0
Vehicles_Involved,0


### Cell 7 — Calculate traffic safety measures

In [ ]:
total_accidents = len(df)
total_injuries = df["Injuries"].sum(min_count=1)
total_fatalities = df["Fatalities"].sum(min_count=1)
severe_count = df["Accident_Severity"].isin(["Severe", "Fatal"]).sum()
severe_share = severe_count / total_accidents * 100 if total_accidents else 0
avg_vehicles = df["Vehicles_Involved"].mean()

print(f"Total accidents: {total_accidents:,}")
print("Reported injuries:", int(total_injuries) if pd.notna(total_injuries) else "Not available")
print("Reported fatalities:", int(total_fatalities) if pd.notna(total_fatalities) else "Not available")
print(f"Severe/fatal accident share: {severe_share:.1f}%")
print("Average vehicles involved:", f"{avg_vehicles:.2f}" if pd.notna(avg_vehicles) else "Not available")
display(df.head())

Total accidents: 1,200
Reported injuries: 1327
Reported fatalities: 63
Severe/fatal accident share: 18.9%
Average vehicles involved: 1.76


,Accident_ID,Accident_Date,City,Area,Road_Type,Accident_Hour,Weather,Vehicle_Type,Accident_Severity,Vehicles_Involved,Injuries,Fatalities,Cause,Month,Year,Weekday,Time_Period
0,ACC00001,2025-05-10,Rajkot,Zone-1,Residential,17,Clear,Car,Minor,2,1,0,Over-Speeding,2025-05,2025,Saturday,Afternoon (12-17)
1,ACC00002,2025-12-31,Gandhinagar,Zone-8,Residential,19,Clear,Two-Wheeler,Severe,4,1,0,Over-Speeding,2025-12,2025,Wednesday,Evening (18-23)
2,ACC00003,2025-11-10,Ahmedabad,Zone-4,Highway,17,Clear,Two-Wheeler,Minor,3,1,0,Signal Violation,2025-11,2025,Monday,Afternoon (12-17)
3,ACC00004,2023-05-02,Rajkot,Zone-4,Residential,6,Clear,Car,Minor,3,1,0,Over-Speeding,2023-05,2023,Tuesday,Morning (06-11)
4,ACC00005,2024-04-11,Surat,Zone-2,Residential,16,Rainy,Car,Minor,3,0,0,Over-Speeding,2024-04,2024,Thursday,Afternoon (12-17)


### Cell 8 — Exploratory Data Analysis (EDA)

In [ ]:
print("Dataset overview")
display(df.describe(include="all").T)

print("\nAccidents by severity")
display(df["Accident_Severity"].value_counts().to_frame("Accidents"))

print("\nAccidents by vehicle type")
display(df["Vehicle_Type"].value_counts().to_frame("Accidents"))

print("\nAccidents by cause")
display(df["Cause"].value_counts().to_frame("Accidents"))

print("\nAccidents by city")
display(df["City"].value_counts().to_frame("Accidents"))

Dataset overview


,count,unique,top,freq,mean,min,25%,50%,75%,max,std
Accident_ID,1200,1200,ACC01184,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Accident_Date,1200,NaN,NaN,NaN,2024-07-15 19:16:48,2023-01-01 00:00:00,2023-09-22 18:00:00,2024-08-15 00:00:00,2025-04-17 00:00:00,2025-12-31 00:00:00,NaN
City,1200,5,Ahmedabad,432,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Area,1200,12,Zone-10,122,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Road_Type,1200,4,Main Road,494,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Accident_Hour,1200.0,NaN,NaN,NaN,13.613333,0.0,9.0,15.0,18.0,23.0,5.639473
Weather,1200,4,Clear,786,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Vehicle_Type,1200,5,Two-Wheeler,491,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Accident_Severity,1200,4,Minor,603,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Vehicles_Involved,1200.0,NaN,NaN,NaN,1.761667,1.0,1.0,2.0,2.0,4.0,0.83193



Accidents by severity


,Accidents
Accident_Severity,
Minor,603
Moderate,370
Severe,184
Fatal,43



Accidents by vehicle type


,Accidents
Vehicle_Type,
Two-Wheeler,491
Car,408
Auto,120
Truck,98
Bus,83



Accidents by cause


,Accidents
Cause,
Over-Speeding,334
Signal Violation,247
Distracted Driving,202
Poor Road Condition,175
Other,142
Weather,100



Accidents by city


,Accidents
City,
Ahmedabad,432
Surat,261
Vadodara,213
Rajkot,188
Gandhinagar,106


### Cell 9 — Create summary tables

In [ ]:
severity_summary = df.groupby("Accident_Severity", as_index=False).size().rename(columns={"size": "Accidents"})
city_summary = df.groupby("City", as_index=False).agg(
    Accidents=("Accident_ID", "count"),
    Injuries=("Injuries", "sum"),
    Fatalities=("Fatalities", "sum")
).sort_values("Accidents", ascending=False)
cause_summary = df.groupby("Cause", as_index=False).size().rename(columns={"size": "Accidents"})
vehicle_summary = df.groupby("Vehicle_Type", as_index=False).size().rename(columns={"size": "Accidents"})
monthly_summary = df.groupby("Month", as_index=False).agg(
    Accidents=("Accident_ID", "count"),
    Injuries=("Injuries", "sum"),
    Fatalities=("Fatalities", "sum")
).sort_values("Month")
hour_summary = df.groupby("Accident_Hour", as_index=False).size().rename(columns={"size": "Accidents"})
road_summary = df.groupby("Road_Type", as_index=False).size().rename(columns={"size": "Accidents"})

display(severity_summary)
display(city_summary)
display(cause_summary)
display(monthly_summary.head())

,Accident_Severity,Accidents
0,Fatal,43
1,Minor,603
2,Moderate,370
3,Severe,184


,City,Accidents,Injuries,Fatalities
0,Ahmedabad,432,462,23
3,Surat,261,271,18
4,Vadodara,213,239,10
2,Rajkot,188,220,6
1,Gandhinagar,106,135,6


,Cause,Accidents
0,Distracted Driving,202
1,Other,142
2,Over-Speeding,334
3,Poor Road Condition,175
4,Signal Violation,247
5,Weather,100


,Month,Accidents,Injuries,Fatalities
0,2023-01,45,45,1
1,2023-02,38,43,0
2,2023-03,31,34,1
3,2023-04,36,43,2
4,2023-05,37,23,2


### Cell 10 — Create charts for initial analysis

In [ ]:
fig1 = px.pie(severity_summary, names="Accident_Severity", values="Accidents",
               title="Accident Severity Distribution", hole=0.35)
fig1.show()

fig2 = px.line(monthly_summary, x="Month", y="Accidents", markers=True,
               title="Monthly Accident Trend")
fig2.update_layout(xaxis_tickangle=-45)
fig2.show()

fig3 = px.bar(city_summary, x="City", y="Accidents", color="Accidents",
              title="Accidents by City")
fig3.show()

fig4 = px.bar(cause_summary, x="Cause", y="Accidents",
              title="Accidents by Reported Cause")
fig4.update_layout(xaxis_tickangle=-35)
fig4.show()

fig5 = px.scatter(df, x="Accident_Hour", y="Injuries", color="Accident_Severity",
                  hover_data=["City", "Road_Type"], title="Accident Hour vs Injuries")
fig5.show()

fig6 = px.bar(hour_summary, x="Accident_Hour", y="Accidents",
              title="Accidents by Hour of Day")
fig6.show()

### Cell 11 — Build the interactive Dash dashboard
Choose a city to update KPI cards and charts.

In [ ]:
app = Dash(__name__)
city_options = [{"label": "All Cities", "value": "All"}] + [
    {"label": str(x), "value": str(x)} for x in sorted(df["City"].dropna().unique())
]

app.layout = html.Div([
    html.H1("Smart City Traffic & Accident Analysis Dashboard",
            style={"textAlign": "center", "color": "#17324D"}),
    html.P("Explore accident patterns, severity, causes and high-risk periods.",
           style={"textAlign": "center"}),
    html.Div([
        html.Label("Select City:"),
        dcc.Dropdown(id="city-filter", options=city_options, value="All",
                     clearable=False, style={"color": "#222", "maxWidth": "500px"})
    ], style={"padding": "15px"}),
    html.Div(id="traffic-kpis",
             style={"display": "flex", "gap": "15px", "flexWrap": "wrap",
                    "justifyContent": "space-around"}),
    html.Div([
        dcc.Graph(id="severity-chart"),
        dcc.Graph(id="monthly-chart"),
        dcc.Graph(id="cause-chart"),
        dcc.Graph(id="hour-chart"),
        dcc.Graph(id="vehicle-chart"),
        dcc.Graph(id="city-chart")
    ], style={"display": "grid",
              "gridTemplateColumns": "repeat(auto-fit, minmax(350px, 1fr))",
              "gap": "12px", "padding": "10px"})
], style={"fontFamily": "Arial", "backgroundColor": "#F4F7FA", "padding": "15px"})

@app.callback(
    Output("traffic-kpis", "children"),
    Output("severity-chart", "figure"),
    Output("monthly-chart", "figure"),
    Output("cause-chart", "figure"),
    Output("hour-chart", "figure"),
    Output("vehicle-chart", "figure"),
    Output("city-chart", "figure"),
    Input("city-filter", "value")
)
def update_traffic_dashboard(selected_city):
    filtered = df if selected_city == "All" else df[df["City"] == selected_city]
    count = len(filtered)
    injuries = filtered["Injuries"].sum(min_count=1)
    fatalities = filtered["Fatalities"].sum(min_count=1)
    severe = filtered["Accident_Severity"].isin(["Severe", "Fatal"]).sum()
    severe_pct = severe / count * 100 if count else 0

    def kpi(title, value):
        return html.Div([
            html.H4(title, style={"marginBottom": "5px"}),
            html.H2(value, style={"marginTop": "0", "color": "#176B87"})
        ], style={"background": "white", "padding": "15px",
                  "borderRadius": "10px", "minWidth": "170px",
                  "boxShadow": "0 1px 5px #D5DDE5"})

    cards = [
        kpi("Total Accidents", f"{count:,}"),
        kpi("Reported Injuries", "N/A" if pd.isna(injuries) else f"{injuries:,.0f}"),
        kpi("Reported Fatalities", "N/A" if pd.isna(fatalities) else f"{fatalities:,.0f}"),
        kpi("Severe/Fatal Share", f"{severe_pct:.1f}%")
    ]

    severity = filtered.groupby("Accident_Severity", as_index=False).size().rename(columns={"size": "Accidents"})
    f1 = px.pie(severity, names="Accident_Severity", values="Accidents",
                title="Accident Severity", hole=0.35)

    month = filtered.groupby("Month", as_index=False).size().rename(columns={"size": "Accidents"}).sort_values("Month")
    f2 = px.line(month, x="Month", y="Accidents", markers=True, title="Monthly Accident Trend")
    f2.update_layout(xaxis_tickangle=-45)

    cause = filtered.groupby("Cause", as_index=False).size().rename(columns={"size": "Accidents"})
    f3 = px.bar(cause, x="Cause", y="Accidents", title="Accidents by Cause")
    f3.update_layout(xaxis_tickangle=-35)

    hour = filtered.groupby("Accident_Hour", as_index=False).size().rename(columns={"size": "Accidents"})
    f4 = px.bar(hour, x="Accident_Hour", y="Accidents", title="Accidents by Hour")

    vehicle = filtered.groupby("Vehicle_Type", as_index=False).size().rename(columns={"size": "Accidents"})
    f5 = px.bar(vehicle, x="Vehicle_Type", y="Accidents", title="Accidents by Vehicle Type")

    city = filtered.groupby("City", as_index=False).size().rename(columns={"size": "Accidents"})
    f6 = px.bar(city, x="City", y="Accidents", title="Accidents by City")

    for fig in [f1, f2, f3, f4, f5, f6]:
        fig.update_layout(template="plotly_white", margin=dict(l=30, r=20, t=60, b=40))
    return cards, f1, f2, f3, f4, f5, f6

print("Traffic dashboard is ready.")

Traffic dashboard is ready.


### Cell 12 — Run the dashboard
Run this cell in Google Colab to open the interactive dashboard.

In [ ]:
app.run(jupyter_mode="external", debug=False, port=8050)

Dash app running on:
Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

### Cell 13 — Generate actionable insights

In [ ]:
if len(df):
    top_city = df["City"].value_counts().idxmax()
    top_cause = df["Cause"].value_counts().idxmax()
    top_vehicle = df["Vehicle_Type"].value_counts().idxmax()
    top_hour = int(df["Accident_Hour"].value_counts().idxmax())
    top_road = df["Road_Type"].value_counts().idxmax()
    severe_fatal = df["Accident_Severity"].isin(["Severe", "Fatal"]).sum()
    severe_pct = severe_fatal / len(df) * 100

    print("ACTIONABLE INSIGHTS")
    print(f"1. City with the most recorded accidents: {top_city}")
    print(f"2. Most frequently reported cause: {top_cause}")
    print(f"3. Most represented vehicle type: {top_vehicle}")
    print(f"4. Hour with the most recorded accidents: {top_hour}:00")
    print(f"5. Most represented road type: {top_road}")
    print(f"6. Severe/fatal accident share: {severe_pct:.1f}%")

    print("\nSuggested actions:")
    print("- Review road-safety conditions and traffic management in locations with consistently high counts.")
    print("- Study peak accident hours before planning traffic awareness or enforcement activities.")
    print("- Evaluate road design, signage and lighting where supported by local field assessments.")
    print("- Treat reported causes as recorded categories, not proven causal explanations.")
else:
    print("No records available for insights.")

ACTIONABLE INSIGHTS
1. City with the most recorded accidents: Ahmedabad
2. Most frequently reported cause: Over-Speeding
3. Most represented vehicle type: Two-Wheeler
4. Hour with the most recorded accidents: 18:00
5. Most represented road type: Main Road
6. Severe/fatal accident share: 18.9%

Suggested actions:
- Review road-safety conditions and traffic management in locations with consistently high counts.
- Study peak accident hours before planning traffic awareness or enforcement activities.
- Evaluate road design, signage and lighting where supported by local field assessments.
- Treat reported causes as recorded categories, not proven causal explanations.


### Cell 14 — Ethical considerations and limitations

Include these points in the report:

1. **Privacy:** Remove names, phone numbers, vehicle registration numbers and other personal identifiers.
2. **Data security:** Do not publish identifiable accident or victim-level records.
3. **Fairness:** Avoid stigmatizing neighbourhoods or road users based on incomplete data.
4. **Data quality:** Document missing values, duplicate records, date errors and inconsistent severity labels.
5. **Transparency:** Clearly state whether the data is official, publicly sourced, collected with permission or synthetic.
6. **Responsible interpretation:** An association between weather, road type or time and accidents does not prove causation.
7. **Sensitive reporting:** Report injuries and fatalities respectfully and only at an appropriate aggregate level.

**Limitations:** Accident records may be under-reported, location labels may be inconsistent, and reporting practices can differ by city. A high accident count does not necessarily mean a location has the highest risk unless traffic volume or exposure data is also available. Synthetic practice data cannot support real-world safety conclusions.

### Cell 15 — Export cleaned data and summaries

In [ ]:
df.to_csv("cleaned_traffic_accident_data.csv", index=False)
severity_summary.to_csv("traffic_severity_summary.csv", index=False)
city_summary.to_csv("traffic_city_summary.csv", index=False)
cause_summary.to_csv("traffic_cause_summary.csv", index=False)
monthly_summary.to_csv("traffic_monthly_summary.csv", index=False)
print("Exported cleaned data and summary CSV files.")

# In Google Colab, uncomment the next line to download the cleaned dataset:
# files.download("cleaned_traffic_accident_data.csv")

Exported cleaned data and summary CSV files.


## Conclusion

The Smart City Traffic & Accident Analysis project uses Python to clean and analyze traffic accident records and uses interactive charts to explore accident severity, locations, causes, vehicle types and time patterns. The dashboard can help decision-makers identify areas for further investigation and plan road-safety improvements. Findings must be interpreted carefully, with attention to privacy, data quality, bias and the difference between correlation and causation.